**실행 설정**

입력은 `config/paths.local.json` 또는 `SLC7A5_DATA_ROOT`로 지정합니다. 결과는 이 저장소의 `results/`에 기록합니다. 입력/실행 설명은 저장소 루트 README를 참고하세요. 이 노트북에는 기존 탐색/대체 Figure 셀이 포함되어 있으며 전체 Figure 재현 검증은 별도 단계입니다.

In [ ]:
# Repository paths; inputs are read-only, new files stay in results/.
from pathlib import Path
import os
import sys
_start = Path(os.environ.get("SLC7A5_REPO_ROOT", str(Path.cwd()))).resolve()
_repo = next((p for p in [_start, *_start.parents] if (p / "slc7a5_paths.py").is_file()), None)
if _repo is None:
    raise RuntimeError("Open this notebook inside the repository or set SLC7A5_REPO_ROOT.")
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from slc7a5_paths import artifact_path, dataset_path, input_path, output_path, output_dir, setup_notebook
setup_notebook('figure3_cellphone')


In [ ]:
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import os
import pickle

from cellphonedb.src.core.methods import cpdb_degs_analysis_method
import ktplotspy as kpy

In [ ]:
import matplotlib as mpl

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["pdf.use14corefonts"] = False

# Final celltype annotate adata

In [ ]:
adata=sc.read(artifact_path('02_SLC7A5_mono_Journal/03_Script/Journal_script/scRNA_monoMAC_detail.h5ad'))
adata

In [ ]:
adata_myl=sc.read(artifact_path('MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono_cellannot.h5ad'))
adata_myl

In [ ]:
sc.pl.umap(adata, color=['celltype'])

In [ ]:
# Lymphoid subset — Leiden re-clustering & annotation

In [ ]:
# ── 1. Lymphoid subset — 원본 UMAP/embedding 유지 ─────────────
adata_lymph = adata[adata.obs['celltype'] == 'Lymphoid'].copy()
print(f"Lymphoid cells: {adata_lymph.n_obs}")

# X_pca_harmony, X_umap은 원본 adata에서 그대로 가져옴
# neighbors만 subset 세포 기준으로 재계산
sc.pp.neighbors(adata_lymph, use_rep='X_pca_harmony', n_neighbors=15)

# UMAP은 재계산하지 않고 원본 좌표 그대로 사용
sc.pl.umap(adata_lymph, color=['detail_cluster', 'sample'], frameon=False)

In [ ]:
# ── 2. Leiden — 여러 resolution 비교 ───────────────────────────
for res in [0.2, 0.4, 0.6, 0.8]:
    sc.tl.leiden(adata_lymph, resolution=res, key_added=f'leiden_{res}')

sc.pl.umap(
    adata_lymph,
    color=['leiden_0.2', 'leiden_0.4', 'leiden_0.6', 'leiden_0.8'],
    ncols=2, legend_loc='on data', frameon=False
)

In [ ]:
# ── 3. Resolution 선택 (결과 보고 조정) ────────────────────────
RESOLUTION = 0.6

adata_lymph.obs['leiden'] = adata_lymph.obs[f'leiden_{RESOLUTION}']
print("Leiden cluster sizes:")
print(adata_lymph.obs['leiden'].value_counts().sort_index())

In [ ]:
# ── 4. 클러스터별 DEG (vs rest, Wilcoxon) ──────────────────────
sc.tl.rank_genes_groups(
    adata_lymph,
    groupby='leiden',
    method='wilcoxon',
    use_raw=True,
    pts=True,
    key_added='rank_leiden'
)

deg_lymph = sc.get.rank_genes_groups_df(adata_lymph, group=None, key='rank_leiden')
print(deg_lymph.head(10))

In [ ]:
# ── 5. 클러스터별 Top 10 DEG 출력 ──────────────────────────────
import pandas as pd

clusters = sorted(adata_lymph.obs['leiden'].unique(), key=lambda x: int(x))
for cl in clusters:
    top = (deg_lymph[deg_lymph['group'] == cl]
           .query('pvals_adj < 0.05 and logfoldchanges > 0')
           .sort_values('scores', ascending=False)
           .head(10)[['names', 'scores', 'logfoldchanges', 'pct_nz_group', 'pct_nz_reference']])
    print(f"\n=== Cluster {cl} top DEGs ===")
    print(top.to_string(index=False))

In [ ]:
# ── 6. Known marker gene dotplot ───────────────────────────────
markers_known = {
    'NK'  : ['NKG7', 'GNLY', 'KLRD1', 'FGFBP2', 'FCGR3A', 'NCAM1'],
    'ILC' : ['IL7R', 'RORC', 'KIT', 'PLCG2'],
    'CD4' : ['CD4', 'IL7R', 'CD3D', 'CD3E', 'MAL'],
    'CD8' : ['CD8A', 'CD8B', 'GZMK', 'GZMB', 'PRF1'],
    'Treg': ['FOXP3', 'IL2RA', 'CTLA4', 'IKZF2'],
}

# var_names에 있는 유전자만 필터링
markers_filtered = {
    k: [g for g in v if g in adata_lymph.var_names]
    for k, v in markers_known.items()
}

sc.pl.dotplot(
    adata_lymph,
    var_names=markers_filtered,
    groupby='leiden',
    standard_scale='var',
    cmap='Blues',
    figsize=(18, len(adata_lymph.obs['leiden'].unique()) * 0.55 + 1),
)

In [ ]:
# ── 7. Top DEG heatmap (클러스터간 비교) ───────────────────────
# leiden_colors가 클러스터 수와 안 맞으면 IndexError 발생 → 초기화 후 재설정
if 'leiden_colors' in adata_lymph.uns:
    del adata_lymph.uns['leiden_colors']

sc.pl.rank_genes_groups_heatmap(
    adata_lymph,
    n_genes=5,
    groupby='leiden',
    key='rank_leiden',
    use_raw=True,
    swap_axes=True,
    vmin=-3, vmax=3,
    cmap='bwr',
    show_gene_labels=True,
    figsize=(14, 8),
    dendrogram=False,
)

In [ ]:
sc.pl.umap(adata_lymph, color=['FOXP3'])

In [ ]:
deg_lymph.to_csv(output_path('02_SLC7A5_mono_Journal/02_Analysis/lym_deg.csv'))

In [ ]:
# ── 8. 수동 annotation ──────────────────────────────────────────
# DEG & dotplot 확인 후 아래 dict 채우기
cluster_annotation = {
    '0': 'CD8',
    '1': 'NK',
    '2': 'CD4',
    '3': 'CD4',
    '4': 'NK',
    '5': 'CD8',
    '6': 'CD8',
    '7': 'ambient',
    '8': 'proliferating T/NK'
}

adata_lymph.obs['lymphoid_type'] = (
    adata_lymph.obs['leiden']
               .map(cluster_annotation)
               .fillna('Unknown')
)

print("lymphoid_type distribution:")
print(adata_lymph.obs['lymphoid_type'].value_counts())

In [ ]:
# ── 9. 최종 UMAP 확인 ──────────────────────────────────────────
palette = {
    'CD8'              : '#3498DB',
    'NK'               : '#E74C3C',
    'CD4'              : '#2ECC71',
    'ambient'          : '#BDC3C7',
    'proliferating T/NK': '#9B59B6',
}
adata_lymph.obs['lymphoid_type'] = pd.Categorical(
    adata_lymph.obs['lymphoid_type'],
    categories=[k for k in palette if k in adata_lymph.obs['lymphoid_type'].unique()]
)
adata_lymph.uns['lymphoid_type_colors'] = [
    palette[k] for k in adata_lymph.obs['lymphoid_type'].cat.categories
]

sc.pl.umap(adata_lymph,
           color=['leiden', 'lymphoid_type'],
           legend_loc='on data', frameon=False, ncols=2)

In [ ]:
adata.obs['lym_cluster']=adata_lymph.obs['lymphoid_type']

In [ ]:
adata.obs['myl_cluster']=adata_myl.obs['monocluster']

In [ ]:
# ── specific_type 생성 ───────────────────────────────────────────
# Lymphoid → lymphoid_type, Myeloid → myl_cluster, 나머지 → celltype

adata.obs['specific_type'] = adata.obs['celltype'].astype(str)

lymphoid_mask = adata.obs['celltype'] == 'Lymphoid'
myeloid_mask  = adata.obs['celltype'] == 'Myeloid'

adata.obs.loc[lymphoid_mask, 'specific_type'] = adata.obs.loc[lymphoid_mask, 'lym_cluster'].astype(str)
adata.obs.loc[myeloid_mask,  'specific_type'] = adata.obs.loc[myeloid_mask,  'myl_cluster'].astype(str)

print("specific_type distribution:")
print(adata.obs['specific_type'].value_counts())

In [ ]:
adata.obs['specific_type'] = adata.obs['specific_type'].replace({
    'mono0': 'Classical_monocyte',
    'mono1': 'Kupffer_cell',
    'mono2': 'cDC2',
    'mono3': 'Non-classical_monocyte',
    'mono4': 'LAM',
    'mono5': 'Special_monocyte',
    'mono6': 'ambient',
    'mono7': 'mo_Kupffer_cell',
    'mono8': 'cDC1',
    'mono9': 'mig_DC'
})

sc.pl.umap(adata, color=['specific_type'])

In [ ]:
adata_final = adata[adata.obs['specific_type'] != 'ambient'].copy()
adata_final

In [ ]:
adata_final.obs['state']

In [ ]:
sc.pl.umap(adata_final, color=['specific_type'])

In [ ]:
adata_final.obs['cluster_state'] = adata_final.obs['specific_type'].astype(str) + '-' + adata_final.obs['state'].astype(str)
adata_final.obs['cluster_state'].value_counts()

In [ ]:
adata_final.write(output_path('02_SLC7A5_mono_Journal/03_Script/Journal_script/scRNA_Final_cellannot.h5ad'))

# method3 cellphoneDB analysis

In [ ]:
adata=sc.read(artifact_path('02_SLC7A5_mono_Journal/03_Script/Journal_script/scRNA_Final_cellannot.h5ad'))

In [ ]:
sc.pl.umap(adata, color=['specific_type'])

In [ ]:
adata.obs['specific_type']

In [ ]:
sc.pl.umap(adata, color = 'specific_type')

In [ ]:
adata=sc.read(artifact_path('02_SLC7A5_mono_Journal/03_Script/Journal_script/scRNA_Final_cellannot.h5ad'))
adata

In [ ]:
adata=sc.read(artifact_path('02_SLC7A5_mono_Journal/03_Script/Journal_script/scRNA_Final_cellannot.h5ad'))
adata

In [ ]:
adata_raw=sc.read(artifact_path('MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_raw.h5ad'))
adata_raw

In [ ]:
common_cells = adata_raw.obs.index.intersection(adata.obs.index)                                                                          
print(f"adata     : {adata.n_obs} cells")                                                                                                 
print(f"adata_raw : {adata_raw.n_obs} cells")                                                                                             
print(f"공통 cells: {len(common_cells)} cells")

In [ ]:
adata_raw = adata_raw[common_cells].copy()
adata_raw

In [ ]:
adata_raw.write(output_path('02_SLC7A5_mono_Journal/03_Script/Journal_script/scRNA_Final_cellannot_raw.h5ad'))

In [ ]:
# ── 경로 설정 ─────────────────────────────────────────────────────────────────
BASE      = output_dir('02_SLC7A5_mono_Journal')
DATA_DIR  = os.path.join(BASE, '01_Data')
ANAL_DIR  = os.path.join(BASE, '02_Analysis')
FIG_DIR   = os.path.join(BASE, '05_Figure')
CPDB_DIR  = os.path.join(ANAL_DIR, 'cellphoneDB')
OUT_DIR   = os.path.join(CPDB_DIR, 'method3')

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# ── CellPhoneDB 입력 파일 경로 ────────────────────────────────────────────────
CPDB_ZIP   = dataset_path('cpdb_database')
META_TSV   = os.path.join(CPDB_DIR, 'cellseq_total_cellphneDB.tsv')
COUNTS_H5AD = artifact_path('02_SLC7A5_mono_Journal/03_Script/Journal_script/scRNA_Final_cellannot_raw.h5ad')
DEGS_TXT   = os.path.join(CPDB_DIR, 'degs_mono3_MASLD_vs_normal.txt')

# ── 분석용 adata 경로 (cluster_state obs 컬럼이 있는 processed adata) ─────────
ADATA_PATH = artifact_path('02_SLC7A5_mono_Journal/03_Script/Journal_script/scRNA_Final_cellannot.h5ad')


# ── 결과 저장 경로 ─────────────────────────────────────────────────────────────
PKL_PATH        = os.path.join(OUT_DIR, 'cpdb_method3_results.pkl')
ADATA_OUT       = os.path.join(DATA_DIR, 'sctotal_01_cpdb_scored.h5ad')
SCORE_CSV_OUT   = os.path.join(OUT_DIR, 'percell_cci_scores.csv')

# ── 관심 클러스터 (per-cell score를 뽑을 클러스터, None이면 전체) ─────────────
# 예: FOCUS_CLUSTERS = ['mono3-Healthy', 'mono3-MASL']
FOCUS_CLUSTERS = None

# ── metadata의 cell-type 컬럼명 ──────────────────────────────────────────────
CELLTYPE_COL = 'cluster_state'   # adata.obs에도 동일 컬럼이 있어야 함

# ── cutree (클러스터 수) ─────────────────────────────────────────────────────
N_CLUSTERS_CUTOFF = 6    # relevant_interaction 필터링 기준 (None = 필터 없이 전체 사용)

In [ ]:
# META_TSV 생성 — Cell barcode + cell type (cluster_state) 매핑                                                                           
metadata = adata.obs[['cluster_state']].reset_index()                                                                                     
metadata.columns = ['Cell', 'cluster_state']                                                                                              

# CellPhoneDB는 탭 구분, 첫 컬럼이 'Cell' 이어야 함                                                                                       
metadata.to_csv(META_TSV, sep='\t', index=False)                                                                                          

print(f"Saved: {META_TSV}")
print(f"Shape: {metadata.shape}")                                                                                                         
print(metadata['cluster_state'].value_counts())

In [ ]:
# ── metadata 확인 ──────────────────────────────────────────────────────────────
metadata = pd.read_csv(META_TSV, sep='\t')
print('metadata shape:', metadata.shape)
print('columns:', metadata.columns.tolist())
print()
print('cluster_state 분포:')
print(metadata[CELLTYPE_COL].value_counts())

In [ ]:
# ── DEG 파일 생성 (CellPhoneDB method 3 형식) ────────────────────────────────
# Cell 30에서 로드한 adata (scRNA_Final_cellannot.h5ad) 사용
# cluster_state 컬럼이 있는지 확인 후 DEG 계산

print("cluster_state 컬럼 존재:", 'cluster_state' in adata.obs.columns)
print(adata.obs['cluster_state'].value_counts())

sc.tl.rank_genes_groups(
    adata,
    groupby='cluster_state',
    method='wilcoxon',
    use_raw=True,
    pts=True,
)

deg_all = sc.get.rank_genes_groups_df(adata, group=None)

# 유의한 DEG 필터링
deg_sig = deg_all[
    (deg_all['pvals_adj'] < 0.05) &
    (deg_all['logfoldchanges'] > 0.25)
].copy()

# CellPhoneDB 형식: cluster, gene 2컬럼
degs_cpdb = deg_sig[['group', 'names']].copy()
degs_cpdb.columns = ['cluster', 'gene']
degs_cpdb = degs_cpdb.drop_duplicates()

# cluster명이 META_TSV의 cluster_state와 일치하는지 확인
print("\nDEG cluster 목록:")
print(degs_cpdb['cluster'].value_counts())

# META_TSV cluster와 일치 여부 검증
meta_clusters = set(pd.read_csv(META_TSV, sep='\t')['cluster_state'].unique())
deg_clusters  = set(degs_cpdb['cluster'].unique())
mismatch = deg_clusters - meta_clusters
if mismatch:
    print(f"\n[경고] META_TSV에 없는 cluster: {mismatch}")
else:
    print("\n[OK] 모든 cluster가 META_TSV와 일치합니다.")

degs_cpdb.to_csv(DEGS_TXT, sep='\t', index=False)
print(f"\nSaved: {DEGS_TXT}  shape={degs_cpdb.shape}")

In [ ]:
# ── DEG 파일 확인 ──────────────────────────────────────────────────────────────
degs_df = pd.read_csv(DEGS_TXT, sep='\t')
print('DEG file shape:', degs_df.shape)
print('columns:', degs_df.columns.tolist())
degs_df.head(3)

In [ ]:
# ── 분석용 adata 로딩 ─────────────────────────────────────────────────────────
print(adata)
print()
print(f"'{CELLTYPE_COL}' obs 컬럼 존재:", CELLTYPE_COL in adata.obs.columns)

if CELLTYPE_COL in adata.obs.columns:
    print('\nadata cluster_state 분포:')
    print(adata.obs[CELLTYPE_COL].value_counts())

In [ ]:
# ── metadata barcode vs adata index 일치 확인 ─────────────────────────────────
meta_barcodes = set(metadata['Cell'].tolist())
adata_barcodes = set(adata.obs_names.tolist())

overlap = meta_barcodes & adata_barcodes
print(f'metadata 바코드: {len(meta_barcodes)}')
print(f'adata 바코드:     {len(adata_barcodes)}')
print(f'겹치는 바코드:    {len(overlap)}')

# metadata와 adata 인덱스가 다를 경우 adata를 metadata 순서로 subset
if len(overlap) < len(meta_barcodes):
    print('\n[주의] 일부 바코드 불일치 — overlap 세포만 사용합니다.')
    adata = adata[adata.obs_names.isin(overlap)].copy()

In [ ]:
# ── Method 3: cpdb_degs_analysis_method ───────────────────────────────────────
# DEG 파일에 의해 어떤 ligand/receptor가 upregulated인지 기반으로 유의 상호작용 선별
# 기존 method1(simple), method2(statistical)과 달리 condition-specific DEG 반영

cpdb_results = cpdb_degs_analysis_method.call(
    cpdb_file_path          = CPDB_ZIP,        # CellphoneDB database zip
    meta_file_path          = META_TSV,        # 바코드 → 클러스터 매핑 tsv
    counts_file_path        = COUNTS_H5AD,     # normalized count matrix (raw h5ad)
    degs_file_path          = DEGS_TXT,        # DEG 파일 (group, names, logfoldchanges ...)
    counts_data             = 'hgnc_symbol',   # gene annotation 형식
    output_path             = OUT_DIR,         # 결과 txt 파일 저장 경로
    output_suffix           = '',
    score_interactions      = True,            # interaction_scores 계산 (필수)
    threshold               = 0.1,             # 각 클러스터에서 발현 세포 비율 threshold
    result_precision        = 3,
    debug                   = False,
)

print('완료! result keys:', list(cpdb_results.keys()))

In [ ]:
# ── 결과 pkl 저장 (재시작 포인트) ────────────────────────────────────────────
with open(PKL_PATH, 'wb') as f:
    pickle.dump(cpdb_results, f)
print('저장 완료:', PKL_PATH)

In [ ]:
# ── 결과 구조 확인 ────────────────────────────────────────────────────────────
for k, v in cpdb_results.items():
    if hasattr(v, 'shape'):
        print(f'{k:35s}  shape={v.shape}')
    else:
        print(f'{k:35s}  type={type(v)}')

# CellphoneDB method3 analysis

In [ ]:
# ── [재시작 시] pkl에서 결과 불러오기 ────────────────────────────────────────
# 위 Section 2 실행을 건너뛰고 싶을 때 이 셀만 실행
with open(artifact_path('02_SLC7A5_mono_Journal/02_Analysis/cellphoneDB/method3/cpdb_method3_results.pkl'), 'rb') as f:
    cpdb_results = pickle.load(f)
print('로드 완료! keys:', list(cpdb_results.keys()))

In [ ]:
cpdb_results

In [ ]:
rel = cpdb_results['relevant_interactions']
sig = cpdb_results['significant_means']
scores = cpdb_results['interaction_scores']

print(rel.shape)   # (interaction 수, cell pair 수)
print(rel.head())

In [ ]:
# ── ktplotspy: 유의 interaction 수 heatmap ──────────────────────────────────── #cell 48
kpy.plot_cpdb_heatmap(
    pvals         = cpdb_results['relevant_interactions'],
    degs_analysis = True,
    title         = 'Sum of significant interactions (Method 3)'
)
plt.savefig(os.path.join(FIG_DIR, 'cpdb_method3_interaction_heatmap.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(FIG_DIR, 'cpdb_method3_interaction_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def make_strength_matrix(condition, cell_pair_cols, sig):
    """특정 condition 내 cell-cell strength matrix 생성"""
    # 양쪽 다 해당 condition인 컬럼만 필터
    target_cols = [c for c in cell_pair_cols 
                   if c.split('|')[0].endswith(f'-{condition}') 
                   and c.split('|')[1].endswith(f'-{condition}')]

    strength = sig[target_cols].apply(pd.to_numeric, errors='coerce').sum(axis=0)

    # cell type 이름에서 condition 제거
    senders   = [c.split('|')[0].rsplit('-', 1)[0] for c in target_cols]
    receivers = [c.split('|')[1].rsplit('-', 1)[0] for c in target_cols]
    cell_types = sorted(set(senders + receivers))

    mat = pd.DataFrame(0.0, index=cell_types, columns=cell_types)
    for col, s, r in zip(target_cols, senders, receivers):
        mat.loc[s, r] = strength[col]

    return mat

cell_pair_cols = [c for c in sig.columns if '|' in c]

mat_healthy = make_strength_matrix('Healthy', cell_pair_cols, sig)
mat_masld   = make_strength_matrix('MASLD',   cell_pair_cols, sig)

# 나란히 비교
fig, axes = plt.subplots(1, 2, figsize=(22, 9))

sns.heatmap(mat_healthy, cmap='Reds',vmin=0, vmax=40, linewidths=0.4, ax=axes[0],
            xticklabels=True, yticklabels=True)
axes[0].set_title('Healthy - interaction strength')
axes[0].tick_params(axis='x', rotation=90, labelsize=8)
axes[0].tick_params(axis='y', rotation=0,  labelsize=8)

sns.heatmap(mat_masld, cmap='Reds', linewidths=0.4, ax=axes[1],
            xticklabels=True, yticklabels=True)
axes[1].set_title('MASLD - interaction strength')
axes[1].tick_params(axis='x', rotation=90, labelsize=8)
axes[1].tick_params(axis='y', rotation=0,  labelsize=8)

plt.tight_layout()
#plt.savefig(os.path.join(FIG_DIR, 'cpdb_strength_healthy_vs_masld.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
print(mat_healthy.index.tolist())
print(mat_masld.index.tolist())

In [ ]:
# Y sender X receiver 

# Difference matrix: MASLD - Healthy
# 양수 = MASLD에서 증가, 음수 = MASLD에서 감소
mat_diff = mat_masld - mat_healthy

fig, axes = plt.subplots(1, 3, figsize=(32, 9))

vmax = max(mat_healthy.values.max(), mat_masld.values.max())

sns.heatmap(mat_healthy, cmap='Reds', vmin=0, vmax=vmax, 
            linewidths=0.4, ax=axes[0], xticklabels=True, yticklabels=True)
axes[0].set_title('Healthy - interaction strength')
axes[0].tick_params(axis='x', rotation=90, labelsize=8)
axes[0].tick_params(axis='y', rotation=0,  labelsize=8)

sns.heatmap(mat_masld, cmap='Reds', vmin=0, vmax=vmax,
            linewidths=0.4, ax=axes[1], xticklabels=True, yticklabels=True)
axes[1].set_title('MASLD - interaction strength')
axes[1].tick_params(axis='x', rotation=90, labelsize=8)
axes[1].tick_params(axis='y', rotation=0,  labelsize=8)

# Diverging colormap: 파랑=감소, 빨강=증가
vabs = max(abs(mat_diff.values.min()), abs(mat_diff.values.max()))
sns.heatmap(mat_diff, cmap='RdBu_r', vmin=-vabs, vmax=vabs,
            linewidths=0.4, ax=axes[2], xticklabels=True, yticklabels=True)
axes[2].set_title('MASLD - Healthy (difference)')
axes[2].tick_params(axis='x', rotation=90, labelsize=8)
axes[2].tick_params(axis='y', rotation=0,  labelsize=8)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'Fig3C_cpdb_strength_diff.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
# 어떤 cell pair가 가장 크게 변했는지 랭킹으로 확인
diff_long = mat_diff.stack().reset_index()
diff_long.columns = ['sender', 'receiver', 'diff']
diff_long['abs_diff'] = diff_long['diff'].abs()
diff_long = diff_long.sort_values('abs_diff', ascending=False)

print("=== MASLD에서 가장 크게 증가한 cell-cell interaction ===")
print(diff_long[diff_long['diff'] > 0].head(10).to_string(index=False))

print("\n=== MASLD에서 가장 크게 감소한 cell-cell interaction ===")
print(diff_long[diff_long['diff'] < 0].head(10).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

# 상위 N개 추출
top_n = 10

increased = diff_long[diff_long['diff'] > 0].head(top_n).copy()
decreased = diff_long[diff_long['diff'] < 0].head(top_n).copy()

# 하나로 합치기 (감소는 역순으로 — 아래쪽에 배치)
plot_df = pd.concat([
    increased.assign(label=increased['sender'] + ' → ' + increased['receiver']),
    decreased.assign(label=decreased['sender'] + ' → ' + decreased['receiver'])
])
plot_df = plot_df.sort_values('diff')  # 작은 값부터 → 위로 갈수록 큰 값

colors = ['#D85A30' if v < 0 else '#1D9E75' for v in plot_df['diff']]

fig, ax = plt.subplots(figsize=(10, 8))

bars = ax.barh(plot_df['label'], plot_df['diff'], color=colors, edgecolor='none', height=0.6)

# 값 레이블
for bar, val in zip(bars, plot_df['diff']):
    x = val + (0.05 if val >= 0 else -0.05)
    ha = 'left' if val >= 0 else 'right'
    ax.text(x, bar.get_y() + bar.get_height() / 2,
            f'{val:+.2f}', va='center', ha=ha, fontsize=9)

# 중앙 기준선
ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')

# 구분선 (증가/감소 사이)
ax.axhline(top_n - 0.5, color='lightgray', linewidth=1)

# 축/레이블
ax.set_xlabel('Interaction strength difference (MASLD - Control)', fontsize=11)
ax.set_title('Top cell-cell interaction changes in MASLD', fontsize=13, fontweight='bold', pad=15)
ax.spines[['top', 'right']].set_visible(False)

# 범례
legend_patches = [
    mpatches.Patch(color='#1D9E75', label='Increased in MASLD'),
    mpatches.Patch(color='#D85A30', label='Decreased in MASLD'),
]
ax.legend(handles=legend_patches, loc='lower right', frameon=False, fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'Fig3Cv2_cell_interaction_diff.pdf'), bbox_inches='tight')
plt.show()

# Special monocyte interaction analysis

In [ ]:
# 관심 있는 cell type pair 컬럼만 추출
# 컬럼명 형식: "CellA|CellB"
target_pairs = [col for col in sig.columns if 'Special_monocyte' in col]

sig_filtered = sig[['interacting_pair'] + target_pairs].copy()
sig_filtered = sig_filtered[sig_filtered[target_pairs].notna().any(axis=1)]
scores_long = scores.set_index('interacting_pair').stack().reset_index()
scores_long.columns = ['interacting_pair', 'cell_pair', 'score']

# 문자열 → float 변환 (변환 불가한 값은 NaN 처리)
scores_long['score'] = pd.to_numeric(scores_long['score'], errors='coerce')

# NaN 제거 후 정렬
scores_long = scores_long.dropna(subset=['score'])
scores_long = scores_long.sort_values('score', ascending=False)

top50 = scores_long.head(50)
print(top50)

In [ ]:
# significant_means를 long format으로
df = sig.set_index('interacting_pair')[target_pairs]
df_long = df.stack().reset_index()
df_long.columns = ['interacting_pair', 'cell_pair', 'mean_expr']

# mean > 0인 것만
df_long = df_long[df_long['mean_expr'] > 0]

# dot plot
fig, ax = plt.subplots(figsize=(35, 40))
scatter = ax.scatter(
    df_long['cell_pair'],
    df_long['interacting_pair'],
    s=df_long['mean_expr'] * 10,
    c=df_long['mean_expr'],
    cmap='Reds'
)
plt.xticks(rotation=90)
plt.colorbar(scatter, label='Mean Expression')
plt.tight_layout()
#plt.savefig('cpdb_dotplot.pdf')
plt.show()

In [ ]:
# sender, receiver 둘 다 같은 condition인 Special_monocyte 관련 pair                                                                      
target_pairs_healthy = [c for c in sig.columns                                                                                            
                        if c.split('|')[0].endswith('-Healthy')                                                                           
                        and c.split('|')[1].endswith('-Healthy')                                                                          
                        and 'Special_monocyte' in c]

target_pairs_masld   = [c for c in sig.columns
                        if c.split('|')[0].endswith('-MASLD')                                                                             
                        and c.split('|')[1].endswith('-MASLD')
                        and 'Special_monocyte' in c]

def make_dotplot(sig, target_pairs, condition, ax):
    df = sig.set_index('interacting_pair')[target_pairs]                                                                                  
    df_long = df.stack().reset_index()
    df_long.columns = ['interacting_pair', 'cell_pair', 'mean_expr']                                                                      
    df_long = df_long[df_long['mean_expr'] > 0]

    # cell pair 이름에서 condition suffix 제거
    df_long['cell_pair'] = df_long['cell_pair'].str.replace(f'-{condition}', '', regex=False)                                             

    vmax = sig[target_pairs_healthy + target_pairs_masld].max().max()                                                                     

    scatter = ax.scatter(                                                                                                                 
        df_long['cell_pair'],
        df_long['interacting_pair'],
        s=df_long['mean_expr'] * 10,                                                                                                      
        c=df_long['mean_expr'],
        cmap='Reds',                                                                                                                      
        vmin=0, vmax=vmax
    )                                                                                                                                     
    ax.set_title(f'Special_monocyte interactions — {condition}', fontsize=12)
    ax.tick_params(axis='x', rotation=90, labelsize=8)                                                                                    
    ax.tick_params(axis='y', labelsize=8)                                                                                                 
    plt.colorbar(scatter, ax=ax, label='Mean Expression')

# y축 항목 수 계산                                                                                                                        
all_pairs = sig.set_index('interacting_pair')[target_pairs_healthy + target_pairs_masld]                                                  
n_lr_pairs = (all_pairs > 0).any(axis=1).sum()                                                                                            

fig_height = max(10, n_lr_pairs * 0.3)  # LR pair 하나당 0.4인치, 최소 10                                                                 

fig, axes = plt.subplots(1, 2, figsize=(24, fig_height), sharey=True)                                                                                

make_dotplot(sig, target_pairs_healthy, 'Healthy', axes[0])                                                                               
make_dotplot(sig, target_pairs_masld,   'MASLD',   axes[1])

plt.tight_layout()
#plt.savefig(os.path.join(FIG_DIR, 'cpdb_Special_mono_dotplot_HvM.pdf'), bbox_inches='tight')                                              
plt.show()  

In [ ]:
# Healthy / MASLD long format 각각 만들기
def to_long(sig, target_pairs, condition):
    df = sig.set_index('interacting_pair')[target_pairs]                                                                                  
    df_long = df.stack().reset_index()
    df_long.columns = ['interacting_pair', 'cell_pair', 'mean_expr']                                                                      
    df_long['cell_pair'] = df_long['cell_pair'].str.replace(f'-{condition}', '', regex=False)                                             
    df_long['mean_expr'] = pd.to_numeric(df_long['mean_expr'], errors='coerce').fillna(0)
    return df_long                                                                                                                        

df_h = to_long(sig, target_pairs_healthy, 'Healthy').rename(columns={'mean_expr': 'expr_healthy'})                                        
df_m = to_long(sig, target_pairs_masld,   'MASLD').rename(columns={'mean_expr': 'expr_masld'})

# merge & diff 계산                                                                                                                       
df_diff = pd.merge(df_h, df_m, on=['interacting_pair', 'cell_pair'], how='outer').fillna(0)                                               
df_diff['diff'] = df_diff['expr_masld'] - df_diff['expr_healthy']                                                                         

# 둘 다 0인 행 제거                                                                                                                       
df_diff = df_diff[(df_diff['expr_healthy'] > 0) | (df_diff['expr_masld'] > 0)]

# y축 정렬: diff 절댓값 기준                                                                                                              
lr_order = (df_diff.groupby('interacting_pair')['diff']
            .apply(lambda x: x.abs().max())                                                                                               
            .sort_values(ascending=True).index.tolist())                                                                                  
df_diff['interacting_pair'] = pd.Categorical(df_diff['interacting_pair'], categories=lr_order, ordered=True)                              
df_diff = df_diff.sort_values('interacting_pair')                                                                                         

# plot                                                                                                                                    
fig_height = max(10, len(lr_order) * 0.35)                                                                                                
fig, ax = plt.subplots(figsize=(14, fig_height))

vabs = df_diff['diff'].abs().max()
scatter = ax.scatter(                                                                                                                     
    df_diff['cell_pair'],
    df_diff['interacting_pair'],                                                                                                          
    s=df_diff['diff'].abs() * 20,   # 크기 = 변화량 절댓값
    c=df_diff['diff'],                                                                                                                    
    cmap='RdBu_r',
    vmin=-vabs, vmax=vabs,                                                                                                                
    edgecolors='grey', linewidths=0.3, alpha=0.85                                                                                         
)                                                                                                                                         
ax.margins(y=0.005)                                                                                                                                         
ax.set_title('MASLD vs Healthy — Special_monocyte interaction change\n(red=increased, blue=decreased in MASLD)',                          
             fontsize=12)
ax.tick_params(axis='x', rotation=90, labelsize=8)                                                                                        
ax.tick_params(axis='y', labelsize=8)
plt.colorbar(scatter, ax=ax, label='Mean expr diff (MASLD - Healthy)')                                                                    
plt.tight_layout()
#plt.savefig(os.path.join(FIG_DIR, 'cpdb_Special_mono_diff_bubbleplot.pdf'), bbox_inches='tight')                                          
plt.show() 

In [ ]:
# ── relevant_interactions: MASLD vs Healthy DEG-support 분류 ─────────────────
# significant_means는 이미 relevant_interactions=1인 경우에만 값이 채워져 있음.
# 즉 df_diff는 이미 DEG-supported interaction만 포함.
# 여기서는 추가로 "어느 condition에서 DEG-support가 있는지"를 분류해
# diff plot에 색으로 overlay함.

RI_raw = cpdb_results['relevant_interactions'].copy()
RI_idx = RI_raw.set_index('interacting_pair') if 'interacting_pair' in RI_raw.columns else RI_raw

# Special_monocyte 관련 Healthy / MASLD 컬럼 분리
ri_h_cols = [c for c in RI_idx.columns
             if c.split('|')[0].endswith('-Healthy')
             and c.split('|')[1].endswith('-Healthy')
             and 'Special_monocyte' in c]

ri_m_cols = [c for c in RI_idx.columns
             if c.split('|')[0].endswith('-MASLD')
             and c.split('|')[1].endswith('-MASLD')
             and 'Special_monocyte' in c]

# LR pair별 각 condition에서 relevant한 cell pair가 하나라도 있는지
ri_h_any = (RI_idx[ri_h_cols] != 0).any(axis=1)
ri_m_any = (RI_idx[ri_m_cols] != 0).any(axis=1)

# 카테고리 분류
cat = pd.Series('None', index=RI_idx.index)
cat[ri_h_any & ri_m_any]  = 'Both'
cat[ri_m_any & ~ri_h_any] = 'MASLD_only'
cat[ri_h_any & ~ri_m_any] = 'Healthy_only'

print("=== Special_monocyte — DEG relevance category ===")
print(cat[cat != 'None'].value_counts())
print(f"\n총 DEG-supported LR pair 수: {(cat != 'None').sum()}")

# df_diff에 카테고리 추가
df_diff_ann = df_diff.copy()
df_diff_ann['rel_category'] = df_diff_ann['interacting_pair'].map(cat).fillna('None')

# ── diff bubble plot: category별 색상 ────────────────────────────────────────
color_map = {
    'MASLD_only':   '#d62728',  # 빨강 — MASLD에서만 DEG-support
    'Healthy_only': '#1f77b4',  # 파랑 — Healthy에서만 DEG-support
    'Both':         '#9467bd',  # 보라 — 양쪽 다 DEG-support
    'None':         '#bbbbbb',  # 회색 — DEG-support 없음 (이론상 없어야 함)
}

fig_height = max(10, len(lr_order) * 0.35)
fig, ax = plt.subplots(figsize=(14, fig_height))

for cat_name, grp in df_diff_ann.groupby('rel_category'):
    ax.scatter(
        grp['cell_pair'],
        grp['interacting_pair'],
        s=grp['diff'].abs() * 20,
        c=color_map.get(cat_name, 'grey'),
        label=cat_name,
        alpha=0.85,
        edgecolors='grey', linewidths=0.3
    )

ax.set_title(
    'MASLD vs Healthy — Special_monocyte interaction change\n'
    '(size=|diff|, color=DEG relevance: red=MASLD_only, blue=Healthy_only, purple=Both)',
    fontsize=11
)
ax.tick_params(axis='x', rotation=90, labelsize=8)
ax.tick_params(axis='y', labelsize=8)
ax.legend(title='DEG relevance', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
plt.tight_layout()
#plt.savefig(os.path.join(FIG_DIR, 'cpdb_Special_mono_diff_by_relevance.pdf'), bbox_inches='tight')
#plt.savefig(os.path.join(FIG_DIR, 'cpdb_Special_mono_diff_by_relevance.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── 요약 테이블: MASLD-specific & diff > 0 (가장 강한 근거) ──────────────────
print("\n=== MASLD_only DEG-support + expression 증가 (top interactions) ===")
strong_masld = (df_diff_ann
                .query("rel_category == 'MASLD_only' and diff > 0")
                .sort_values('diff', ascending=False))
print(strong_masld[['interacting_pair', 'cell_pair', 'expr_healthy', 'expr_masld', 'diff']].to_string(index=False))

print("\n=== Healthy_only DEG-support + expression 감소 (Healthy 특이적) ===")
strong_healthy = (df_diff_ann
                  .query("rel_category == 'Healthy_only' and diff < 0")
                  .sort_values('diff', ascending=True))
print(strong_healthy[['interacting_pair', 'cell_pair', 'expr_healthy', 'expr_masld', 'diff']].to_string(index=False))


In [ ]:
# ── strong_masld / strong_healthy CSV 저장 ────────────────────────────────────
CPDB_OUT = os.path.join(CPDB_DIR, 'method3')

strong_masld = (df_diff_ann
                .query("rel_category == 'MASLD_only' and diff > 0")
                .sort_values('diff', ascending=False))

strong_healthy = (df_diff_ann
                  .query("rel_category == 'Healthy_only' and diff < 0")
                  .sort_values('diff', ascending=True))

strong_masld.to_csv(os.path.join(CPDB_OUT, 'strong_masld_SpecialMono.csv'), index=False)
strong_healthy.to_csv(os.path.join(CPDB_OUT, 'strong_healthy_SpecialMono.csv'), index=False)

print(f"strong_masld  저장: {len(strong_masld)}개 interactions")
print(f"strong_healthy 저장: {len(strong_healthy)}개 interactions")
print("저장 경로:", CPDB_OUT)


In [ ]:
# ── 저장된 파일 로드 & 관심 cell type 필터링 ──────────────────────────────────
CPDB_OUT = os.path.join(CPDB_DIR, 'method3')

strong_masld   = pd.read_csv(os.path.join(CPDB_OUT, 'strong_masld_SpecialMono.csv'))
strong_healthy = pd.read_csv(os.path.join(CPDB_OUT, 'strong_healthy_SpecialMono.csv'))

# ── 관심 파트너 cell type 정의 ────────────────────────────────────────────────
# ※ 데이터 내 실제 표기명 사용 (Stellate → Stellete)
TARGET_PARTNERS = ['CD4', 'CD8', 'NK', 'LAM', 'Stellete', 'Hepatocytes']

def filter_by_partner(df, partners):
    """cell_pair 컬럼에서 Special_monocyte + 지정 파트너 쌍만 추출."""
    mask = df['cell_pair'].apply(
        lambda p: any(
            partner in p.split('|')[0] or partner in p.split('|')[1]
            for partner in partners
        )
    )
    return df[mask].copy()

masld_filtered   = filter_by_partner(strong_masld,   TARGET_PARTNERS)
healthy_filtered = filter_by_partner(strong_healthy, TARGET_PARTNERS)

# sender / receiver 분리 컬럼 추가
masld_filtered[['sender', 'receiver']] = masld_filtered['cell_pair'].str.split('|', expand=True)
healthy_filtered[['sender', 'receiver']] = healthy_filtered['cell_pair'].str.split('|', expand=True)

print("=== MASLD에서 증가한 Special_monocyte interactions (partner 필터) ===")
print(f"총 {len(masld_filtered)}개\n")
print(masld_filtered[['interacting_pair', 'cell_pair', 'sender', 'receiver',
                       'expr_healthy', 'expr_masld', 'diff']].to_string(index=False))

print("\n=== Healthy에서 높은 Special_monocyte interactions (partner 필터) ===")
print(f"총 {len(healthy_filtered)}개\n")
print(healthy_filtered[['interacting_pair', 'cell_pair', 'sender', 'receiver',
                         'expr_healthy', 'expr_masld', 'diff']].to_string(index=False))


In [ ]:
# ── MASLD 증가 interaction dot plot: sender(왼) / receiver(오) 분리 ───────────
if len(masld_filtered) == 0:
    print("필터링 결과 없음 — TARGET_PARTNERS 명칭 확인 필요")
else:
    partner_colors = {
        'CD4':        '#e41a1c',
        'CD8':        '#377eb8',
        'NK':         '#4daf4a',
        'LAM':        '#ff7f00',
        'Stellete':   '#984ea3',
        'Hepatocytes':'#a65628',
    }

    def get_partner(cell_pair, partners):
        for p in cell_pair.split('|'):
            for t in partners:
                if t in p:
                    return t
        return 'other'

    masld_filtered['partner'] = masld_filtered['cell_pair'].apply(
        lambda x: get_partner(x, TARGET_PARTNERS))

    # sender / receiver 분리
    df_sender   = masld_filtered[masld_filtered['sender'].str.contains('Special_monocyte')].copy()
    df_receiver = masld_filtered[masld_filtered['receiver'].str.contains('Special_monocyte')].copy()

    # x축: partner cell type (Special_monocyte 제거)
    df_sender['partner_cell']   = df_sender['receiver']
    df_receiver['partner_cell'] = df_receiver['sender']

    # y축 LR pair 순서: 전체 diff 기준 내림차순
    lr_order = (masld_filtered.groupby('interacting_pair')['diff']
                .max().sort_values(ascending=True).index.tolist())

    for df_ in [df_sender, df_receiver]:
        df_['interacting_pair'] = pd.Categorical(
            df_['interacting_pair'], categories=lr_order, ordered=True)

    fig_h = max(8, len(lr_order) * 0.45)
    fig, axes = plt.subplots(1, 2, figsize=(16, fig_h), sharey=True)

    for ax, df_, title in zip(
        axes,
        [df_sender, df_receiver],
        ['← Sender\nSpecial_monocyte → Partner', 'Receiver →\nPartner → Special_monocyte']
    ):
        for partner, grp in df_.groupby('partner'):
            ax.scatter(
                grp['partner_cell'],
                grp['interacting_pair'],
                s=grp['diff'] * 30,
                c=partner_colors.get(partner, '#999999'),
                label=partner,
                alpha=0.85,
                edgecolors='grey', linewidths=0.3
            )
        ax.set_title(title, fontsize=11)
        ax.tick_params(axis='x', rotation=90, labelsize=9)
        ax.tick_params(axis='y', labelsize=8)
        ax.set_xlabel('Partner cell type', fontsize=10)

    axes[0].set_ylabel('Interacting pair (LR)', fontsize=10)
    axes[1].legend(title='Partner cell type', bbox_to_anchor=(1.02, 1),
                   loc='upper left', fontsize=9)

    fig.suptitle(
        'MASLD-increased Special_monocyte interactions\n'
        '(MASLD_only DEG-support, diff > 0 | bubble size = mean expr diff)',
        fontsize=12, y=1.01
    )
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, 'cpdb_Special_mono_MASLD_increased_targets.pdf'),
                bbox_inches='tight')
    plt.savefig(os.path.join(FIG_DIR, 'cpdb_Special_mono_MASLD_increased_targets.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

    print(f"Sender 방향: {len(df_sender)}개 | Receiver 방향: {len(df_receiver)}개")
    print("저장 완료: cpdb_Special_mono_MASLD_increased_targets")


In [ ]:
# ── Healthy 증가 interaction dot plot: sender(왼) / receiver(오) 분리 ───────────
if len(healthy_filtered) == 0:
    print("필터링 결과 없음 — TARGET_PARTNERS 명칭 확인 필요")
else:
    partner_colors = {
        'CD4':        '#e41a1c',
        'CD8':        '#377eb8',
        'NK':         '#4daf4a',
        'LAM':        '#ff7f00',
        'Stellete':   '#984ea3',
        'Hepatocytes':'#a65628',
    }

    def get_partner(cell_pair, partners):
        for p in cell_pair.split('|'):
            for t in partners:
                if t in p:
                    return t
        return 'other'

    healthy_filtered['partner'] = healthy_filtered['cell_pair'].apply(
        lambda x: get_partner(x, TARGET_PARTNERS))

    # sender / receiver 분리
    df_sender   = healthy_filtered[healthy_filtered['sender'].str.contains('Special_monocyte')].copy()
    df_receiver = healthy_filtered[healthy_filtered['receiver'].str.contains('Special_monocyte')].copy()

    # x축: partner cell type (Special_monocyte 제거)
    df_sender['partner_cell']   = df_sender['receiver']
    df_receiver['partner_cell'] = df_receiver['sender']

    # y축 LR pair 순서: 전체 diff 기준 내림차순
    lr_order = (healthy_filtered.groupby('interacting_pair')['diff']
                .max().sort_values(ascending=True).index.tolist())

    for df_ in [df_sender, df_receiver]:
        df_['interacting_pair'] = pd.Categorical(
            df_['interacting_pair'], categories=lr_order, ordered=True)

    fig_h = max(8, len(lr_order) * 0.45)
    fig, axes = plt.subplots(1, 2, figsize=(16, fig_h), sharey=True)

    for ax, df_, title in zip(
        axes,
        [df_sender, df_receiver],
        ['← Sender\nSpecial_monocyte → Partner', 'Receiver →\nPartner → Special_monocyte']
    ):
        for partner, grp in df_.groupby('partner'):
            ax.scatter(
                grp['partner_cell'],
                grp['interacting_pair'],
                s=grp['diff'].abs() * 30,
                c=partner_colors.get(partner, '#999999'),
                label=partner,
                alpha=0.85,
                edgecolors='grey', linewidths=0.3
            )
        ax.set_title(title, fontsize=11)
        ax.tick_params(axis='x', rotation=90, labelsize=9)
        ax.tick_params(axis='y', labelsize=8)
        ax.set_xlabel('Partner cell type', fontsize=10)

    axes[0].set_ylabel('Interacting pair (LR)', fontsize=10)
    axes[1].legend(title='Partner cell type', bbox_to_anchor=(1.02, 1),
                   loc='upper left', fontsize=9)

    fig.suptitle(
        'Healthy-increased Special_monocyte interactions\n'
        '(Healthy_only DEG-support, diff < 0 | bubble size = |mean expr diff|)',
        fontsize=12, y=1.01
    )
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, 'cpdb_Special_mono_healthy_increased_targets.pdf'),
                bbox_inches='tight')
    plt.savefig(os.path.join(FIG_DIR, 'cpdb_Special_mono_healthy_increased_targets.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

    print(f"Sender 방향: {len(df_sender)}개 | Receiver 방향: {len(df_receiver)}개")
    print("저장 완료: cpdb_Special_mono_healthy_increased_targets")


In [ ]:
print(masld_filtered[['interacting_pair', 'sender', 'receiver', 'diff']].sort_values('diff', ascending=False).to_string())

In [ ]:
strong_healthy

In [ ]:
# ── direction 컬럼 생성 (Sender → Receiver) ──────────────────────────────
masld_filtered['direction'] = (                                                                                                           
    masld_filtered['sender'] + ' → ' + masld_filtered['receiver']                                                                         
)                                                                                                                                         

# ── x축: LR pair 순서 (diff 내림차순) ──────────────────────────────────                                                                 
lr_order = (
    masld_filtered.groupby('interacting_pair')['diff']                                                                                    
    .max().sort_values(ascending=False).index.tolist()                                                                                    
)

# ── y축: direction 순서 ─────────────────────────────────────────────────                                                                
direction_order = sorted(masld_filtered['direction'].unique())

# 숫자 좌표로 변환 (scatter에 categorical 직접 쓰면 순서 보장 안됨)                                                                       
lr_idx      = {v: i for i, v in enumerate(lr_order)}
dir_idx     = {v: i for i, v in enumerate(direction_order)}                                                                               

masld_filtered['x_pos'] = masld_filtered['interacting_pair'].map(lr_idx)                                                                  
masld_filtered['y_pos'] = masld_filtered['direction'].map(dir_idx)                                                                        

# ── 플롯 ────────────────────────────────────────────────────────────────
fig_w = max(10, len(lr_order) * 0.45)                                                                                                     
fig_h = max(4,  len(direction_order) * 0.6)                                                                                               

fig, ax = plt.subplots(figsize=(fig_w, fig_h))                                                                                            

for partner, grp in masld_filtered.groupby('partner'):                                                                                    
    ax.scatter( 
        grp['x_pos'],
        grp['y_pos'],
        s=grp['diff'] * 30,
        c=partner_colors.get(partner, '#999999'),                                                                                         
        label=partner,
        alpha=0.85,                                                                                                                       
        edgecolors='grey', linewidths=0.3
    )

# 축 눈금 설정                                                                                                                            
ax.set_xticks(range(len(lr_order)))
ax.set_xticklabels(lr_order, rotation=90, fontsize=8)                                                                                     
ax.set_yticks(range(len(direction_order)))                                                                                                
ax.set_yticklabels(direction_order, fontsize=9)

ax.set_xlabel('Interacting pair (LR)', fontsize=10)                                                                                       
ax.set_ylabel('Direction (Sender → Receiver)', fontsize=10)
ax.legend(title='Partner cell type', bbox_to_anchor=(1.02, 1),                                                                            
          loc='upper left', fontsize=9)                                                                                                   
ax.grid(axis='x', linestyle='--', alpha=0.3)

fig.suptitle(   
    'MASLD-increased Special_monocyte interactions\n'                                                                                     
    '(bubble size = diff)',
    fontsize=12, y=1.02                                                                                                                   
)
plt.tight_layout()                                                                                                                        
#plt.savefig(os.path.join(FIG_DIR, 'cpdb_Special_mono_direction_dotplot.pdf'),
#            bbox_inches='tight')                                                                                                          
#plt.savefig(os.path.join(FIG_DIR, 'cpdb_Special_mono_direction_dotplot.png'),
#          dpi=150, bbox_inches='tight')                                                                                                 
plt.show()

In [ ]:
# ── direction 정렬: Stellate, Hepatocyte, CD4, CD8, LAM, NK, T/NK 순 ───────
CELL_ORDER = ['Stellate', 'Hepatocyte', 'CD4', 'CD8', 'LAM', 'NK', 'T/NK']

direction_order_sorted= ['Stellete → Special_monocyte','Hepatocytes → Special_monocyte',
                         'Special_monocyte → CD4', 'CD4 → Special_monocyte', 'Special_monocyte → CD8',
                         'CD8 → Special_monocyte', 'Special_monocyte → LAM', 'Special_monocyte → NK','NK → Special_monocyte',
                         'Special_monocyte → proliferating T/NK','proliferating T/NK → Special_monocyte', ]

# 새 y좌표 매핑
dir_idx_sorted = {v: i for i, v in enumerate(direction_order_sorted)}
masld_filtered['y_pos'] = masld_filtered['direction'].map(dir_idx_sorted)

# ── 플롯 (정렬 적용) ────────────────────────────────────────────────────
fig_w = max(10, len(lr_order) * 0.45)
fig_h = max(4,  len(direction_order_sorted) * 0.6)

fig, ax = plt.subplots(figsize=(fig_w, fig_h))

for partner, grp in masld_filtered.groupby('partner'):
    ax.scatter(
        grp['x_pos'],
        grp['y_pos'],
        s=grp['diff'] * 30,
        c=partner_colors.get(partner, '#999999'),
        label=partner,
        alpha=0.85,
        edgecolors='grey', linewidths=0.3
    )

ax.set_xticks(range(len(lr_order)))
ax.set_xticklabels(lr_order, rotation=90, fontsize=8)
ax.set_yticks(range(len(direction_order_sorted)))
ax.set_yticklabels(direction_order_sorted, fontsize=9)

ax.set_xlabel('Interacting pair (LR)', fontsize=10)
ax.set_ylabel('Direction (Sender → Receiver)', fontsize=10)
ax.legend(title='Partner cell type', bbox_to_anchor=(1.02, 1),
          loc='upper left', fontsize=9)
ax.grid(axis='x', linestyle='--', alpha=0.3)

fig.suptitle(
    'MASLD-increased Special_monocyte interactions'
    '(bubble size = diff)',
    fontsize=12, y=1.02
)
plt.tight_layout()
#plt.savefig(os.path.join(FIG_DIR, 'cpdb_Special_mono_direction_dotplot.pdf'),
#            bbox_inches='tight')
#plt.savefig(os.path.join(FIG_DIR, 'cpdb_Special_mono_direction_dotplot.png'),
#            dpi=150, bbox_inches='tight')
plt.show()
print('정렬 순서:', direction_order_sorted)

In [ ]:
masld_filtered

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import os

# ── 카테고리별 LR pair 순서 수동 정렬 ────────────────────────────────────
lr_order_sorted = [
    # Recruitment
    'CCL5_CCR5',
    'LPAR2_ADGRE5',
    'SEMA4D_PLXNB2',
    'SEMA7A_PLXNC1',

    # Anchoring
    'VTN_integrin_aVb1_complex',
    'THY1_integrin_aXb2_complex',
    'THY1_integrin_aMb2_complex',
    'FN1_integrin_aVb1_complex',
    'THBS2_CD36',
    'THY1_ADGRE5',
    'THY1_ADGRE2',
    'CD248_IL6R',
    'DHEAsulfate_bySULT2A1_PPARD',
    'DHEAsulfate_bySULT2B_PPARD',
    'CCL15_CCR1',
    'CCL16_CCR1',
    'Desmosterol_byDHCR7_NR1H2',

    # Immune suppression
    'HLA-B_KIR3DL2',
    'HLA-C_KIR2DL3',
    'HLA-E_KLRK1',
    'HLA-E_KLRC2',
    'HLA-E_CD94:NKG2C',
    'HLA-E_CD94:NKG2E',
    'HLA-F_KIR3DL2',
    'CD86_CTLA4',
    'ENTPD1_ADORA2A',
    'TNFSF12_TNFRSF25',
    'TGM2_ADGRG1',
    'Cholesterol_byLIPA_RORC',
    'VEGFB_NRP1',
    'SEMA4D_CD72',

    # Bidirectional
    'SEMA4D_PTPRC',
]

# ── direction 정렬 ────────────────────────────────────────────────────────
direction_order_sorted = [
    'Stellete → Special_monocyte',
    'Hepatocytes → Special_monocyte',
    'Special_monocyte → CD4',
    'CD4 → Special_monocyte',
    'Special_monocyte → CD8',
    'CD8 → Special_monocyte',
    'Special_monocyte → LAM',
    'Special_monocyte → NK',
    'NK → Special_monocyte',
    'Special_monocyte → proliferating T/NK',
    'proliferating T/NK → Special_monocyte',
]

# ── 좌표 매핑 ─────────────────────────────────────────────────────────────
lr_idx_sorted = {v: i for i, v in enumerate(lr_order_sorted)}
dir_idx_sorted = {v: i for i, v in enumerate(direction_order_sorted)}

# lr_order_sorted에 없는 pair는 제거
masld_filtered = masld_filtered[
    masld_filtered['interacting_pair'].isin(lr_order_sorted)
].copy()

masld_filtered['x_pos'] = masld_filtered['interacting_pair'].map(lr_idx_sorted)
masld_filtered['y_pos'] = masld_filtered['direction'].map(dir_idx_sorted)

# y_pos 매핑 안된 행 제거
masld_filtered = masld_filtered.dropna(subset=['x_pos', 'y_pos'])

# ── 파트너 색상 ───────────────────────────────────────────────────────────
partner_colors = {
    'CD4':        '#e41a1c',
    'CD8':        '#377eb8',
    'NK':         '#4daf4a',
    'LAM':        '#ff7f00',
    'Stellete':   '#984ea3',
    'Hepatocytes':'#a65628',
}

# ── 카테고리 경계 정의 ────────────────────────────────────────────────────
category_boundaries = {
    'Recruitment':        (0,  3,  '#2166ac'),
    'Anchoring':          (4,  16, '#4dac26'),
    'Immune suppression': (17, 33, '#d01c8b'),
    'Bidirectional':      (34, 34, '#808080'),
}

# ── 플롯 ──────────────────────────────────────────────────────────────────
fig_w = max(14, len(lr_order_sorted) * 0.5)
fig_h = max(5,  len(direction_order_sorted) * 0.7)

fig, ax = plt.subplots(figsize=(fig_w, fig_h))

# 카테고리 배경색
for cat, (start, end, color) in category_boundaries.items():
    ax.axvspan(start - 0.5, end + 0.5,
               alpha=0.06, color=color, zorder=0)

# 버블
for partner, grp in masld_filtered.groupby('partner'):
    ax.scatter(
        grp['x_pos'],
        grp['y_pos'],
        s=grp['diff'] * 60,
        c=partner_colors.get(partner, '#999999'),
        label=partner,
        alpha=0.85,
        edgecolors='grey', linewidths=0.3,
        zorder=2
    )

# x축
ax.set_xticks(range(len(lr_order_sorted)))
ax.set_xticklabels(lr_order_sorted, rotation=90, fontsize=8)

# y축
ax.set_yticks(range(len(direction_order_sorted)))
ax.set_yticklabels(direction_order_sorted, fontsize=9)

# 카테고리 라벨 (x축 아래)
for cat, (start, end, color) in category_boundaries.items():
    ax.text((start + end) / 2, -2.2, cat,
            ha='center', fontsize=8, style='italic',
            color=color, fontweight='bold',
            transform=ax.get_xaxis_transform(),
            clip_on=False)

# 카테고리 구분 세로선
for cat, (start, end, color) in category_boundaries.items():
    if end < len(lr_order_sorted) - 1:
        ax.axvline(x=end + 0.5, color='grey',
                   linestyle='--', linewidth=0.8, alpha=0.5)

ax.set_xlabel('Interacting pair (LR)', fontsize=10, labelpad=30)
ax.set_ylabel('Direction (Sender → Receiver)', fontsize=10)
ax.grid(axis='x', linestyle='--', alpha=0.2)

ax.legend(title='Partner cell type',
          bbox_to_anchor=(1.02, 1),
          loc='upper left', fontsize=9)

fig.suptitle(
    'MASLD-increased Special_monocyte interactions\n'
    '(bubble size = mean expr diff)',
    fontsize=12, y=1.02
)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'Fig3Dv1_Special_mono_categorized.pdf'),
            dpi=300,bbox_inches='tight')
#plt.savefig(os.path.join(FIG_DIR, 'cpdb_Special_mono_categorized.png'),
#            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 74 플롯 + bubble size 범례 추가 ────────────────────────────────────
fig_w = max(14, len(lr_order_sorted) * 0.5)
fig_h = max(5,  len(direction_order_sorted) * 0.7)

fig, ax = plt.subplots(figsize=(fig_w, fig_h))

# 카테고리 배경색
for cat, (start, end, color) in category_boundaries.items():
    ax.axvspan(start - 0.5, end + 0.5,
               alpha=0.06, color=color, zorder=0)

# 버블
for partner, grp in masld_filtered.groupby("partner"):
    ax.scatter(
        grp["x_pos"],
        grp["y_pos"],
        s=grp["diff"] * 60,
        c=partner_colors.get(partner, "#999999"),
        label=partner,
        alpha=0.85,
        edgecolors="grey", linewidths=0.3,
        zorder=2
    )

# x축
ax.set_xticks(range(len(lr_order_sorted)))
ax.set_xticklabels(lr_order_sorted, rotation=90, fontsize=8)

# y축
ax.set_yticks(range(len(direction_order_sorted)))
ax.set_yticklabels(direction_order_sorted, fontsize=9)

# 카테고리 라벨 (x축 아래)
for cat, (start, end, color) in category_boundaries.items():
    ax.text((start + end) / 2, -2.2, cat,
            ha="center", fontsize=8, style="italic",
            color=color, fontweight="bold",
            transform=ax.get_xaxis_transform(),
            clip_on=False)

# 카테고리 구분 세로선
for cat, (start, end, color) in category_boundaries.items():
    if end < len(lr_order_sorted) - 1:
        ax.axvline(x=end + 0.5, color="grey",
                   linestyle="--", linewidth=0.8, alpha=0.5)

ax.set_xlabel("Interacting pair (LR)", fontsize=10, labelpad=30)
ax.set_ylabel("Direction (Sender → Receiver)", fontsize=10)
ax.grid(axis="x", linestyle="--", alpha=0.2)

# ── Partner cell type 범례 ──────────────────────────────────────────────────
partner_legend = ax.legend(
    title="Partner cell type",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=9
)
ax.add_artist(partner_legend)

# ── Bubble size 범례 ────────────────────────────────────────────────────────
diff_values = masld_filtered["diff"].dropna()
size_ref = sorted(set([
    round(diff_values.min(), 2),
    round(diff_values.median(), 2),
    round(diff_values.max(), 2),
]))
size_handles = [
    plt.scatter([], [], s=v * 60, c="grey", alpha=0.7,
                edgecolors="grey", linewidths=0.3, label=f"{v:.2f}")
    for v in size_ref
]
ax.legend(
    handles=size_handles,
    title="diff (mean expr)",
    bbox_to_anchor=(1.02, 0.5),
    loc="upper left",
    fontsize=9,
    frameon=True
)

fig.suptitle(
    "MASLD-increased Special_monocyte interactions"
    "(bubble size = mean expr diff)",
    fontsize=12, y=1.02
)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Fig3Dv2_Special_mono_categorized_sizelegend.pdf"),
            dpi=300, bbox_inches="tight")
#plt.savefig(os.path.join(FIG_DIR, "Fig3Dv2_Special_mono_categorized_sizelegend.png"),
#            dpi=150, bbox_inches="tight")
plt.show()